In [1]:
# Parameters
USE_SPACY = "true"
DATA_DIR = "."
TRAIN_CSV = "train.csv"
VAL_CSV = "val.csv"
TEST_CSV = "test.csv"
SAVE_DIR = "/home/santoshd/hvsm/weights"
MAX_TRAIN_TIME_PER_MODEL_MIN = 160
EARLY_DROP_MIN = 25
EARLY_DROP_F1 = 0.68
N_FOLDS = 3
MAX_SEQ_LEN = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
GRAD_ACCUM_STEPS = 2
MIXED_PRECISION = "bf16"
GRADIENT_CHECKPOINTING = "true"
PIN_MEMORY = "true"
NUM_WORKERS = 4
PREFETCH_FACTOR = 2
PERSISTENT_WORKERS = "true"
DATALOADER_MEMORY_EFFICIENT = "true"
KEEP_IN_MEMORY = "false"
SAVE_ONLY_BEST = "true"
SAVE_EVERY_FOLD = "true"
LOG_ARTIFACTS_TO_NOTEBOOK = "false"
PLOT_LEVEL = "light"
BASE_MODEL_NAME = "distilbert-base-uncased"
TFIDF_MAX_FEATS = 50000
TEXT_COL = "text"
LABEL_COL = "label"
RANDOM_SEED = 42


# HVSM — TF–IDF + LR/XGB with Binary Rules, CV, and Prevalence Match

Inputs: `train.csv`, `val.csv`, `test.csv` in this folder. `test.csv` must have `id` and no `label`. Output: `submission.csv`.

## Imports and guardrails

In [2]:
from __future__ import annotations
import os, re, string, warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import numpy as np, pandas as pd
from tqdm import tqdm
from scipy import stats
from scipy.sparse import csr_matrix, hstack, vstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve, precision_recall_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except Exception:
    sns = None
try:
    from textblob import TextBlob
except Exception:
    TextBlob = None
    warnings.warn('TextBlob missing; sentiment features set to zeros.')
np.set_printoptions(linewidth=79)
pd.set_option('display.width', 79)
pd.set_option('display.max_columns', 60)
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


/tmp/ipykernel_2079016/1003607189.py:25: UserWarning: TextBlob missing; sentiment features set to zeros.
  warnings.warn('TextBlob missing; sentiment features set to zeros.')


## Configuration

In [3]:
@dataclass
class Config:
    tfidf_max_features: int = 50000
    tfidf_ngram_max: int = 3
    use_char_ngrams: bool = False
    char_tfidf_max_features: int = 20000
    min_df: int = 2
    kfolds: int = 5
    xgb_iter: int = 25
    lr_iter: int = 25
    plot_level: str = 'full'
CFG = Config(); print(CFG)


Config(tfidf_max_features=50000, tfidf_ngram_max=3, use_char_ngrams=False, char_tfidf_max_features=20000, min_df=2, kfolds=5, xgb_iter=25, lr_iter=25, plot_level='full')


## Plotting helpers

In [4]:
def _tight() -> None:
    plt.tight_layout()
def qq_plot(residuals: np.ndarray, title: str) -> None:
    plt.figure(figsize=(5, 4)); stats.probplot(residuals, dist='norm',
                                               plot=plt)
    plt.title(title); _tight(); plt.show()
def residual_plot(y_true: np.ndarray, y_prob: np.ndarray,
                  title: str) -> None:
    resid = y_true - y_prob
    plt.figure(figsize=(5, 4)); plt.scatter(y_prob, resid, s=8)
    plt.axhline(0.0, linestyle='--'); plt.xlabel('p(y=1)');
    plt.ylabel('residual'); plt.title(title); _tight(); plt.show()
def violin_by_label(df: pd.DataFrame, label_col: str, feat_col: str,
                    title: str) -> None:
    if sns is None:
        df.boxplot(column=feat_col, by=label_col, figsize=(5, 4))
        plt.title(title); plt.suptitle(''); _tight(); plt.show(); return
    plt.figure(figsize=(5, 4));
    sns.violinplot(data=df, x=label_col, y=feat_col)
    plt.title(title); _tight(); plt.show()
def plot_roc_pr(y_true: np.ndarray, y_prob: np.ndarray, title: str)->None:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(fpr, tpr)
    ax[0].set_title(f'ROC AUC={roc_auc_score(y_true, y_prob):.3f}')
    ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
    ax[1].plot(rec, prec); ax[1].set_title('Precision–Recall')
    ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
    _tight(); plt.show()
def plot_confusion(y_true: np.ndarray, y_hat: np.ndarray, title: str)->None:
    cm = confusion_matrix(y_true, y_hat)
    plt.figure(figsize=(4, 3)); plt.imshow(cm, cmap='Blues')
    plt.title(title); plt.colorbar()
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, int(cm[i, j]), ha='center', va='center')
    plt.xlabel('Pred'); plt.ylabel('True'); _tight(); plt.show()


## Processing and features

In [5]:
def _ttr(text: str) -> float:
    toks = re.findall(r'\S+', text.lower())
    return float(len(set(toks)) / len(toks)) if toks else 0.0
def _sentiment(df: pd.DataFrame) -> pd.DataFrame:
    if TextBlob is None:
        df['sentiment_polarity'] = 0.0
        df['sentiment_subjectivity'] = 0.0
        return df
    tqdm.pandas();
    df['sentiment_polarity'] = df['text'].progress_apply(
        lambda x: float(TextBlob(x).sentiment.polarity))
    df['sentiment_subjectivity'] = df['text'].progress_apply(
        lambda x: float(TextBlob(x).sentiment.subjectivity))
    return df
def process_text_file(filename: str) -> pd.DataFrame:
    df = pd.read_csv(os.path.join(filename))
    assert 'text' in df.columns
    df['text'] = df['text'].astype(str)
    df['text_length'] = df['text'].str.len()
    df['word_count'] = df['text'].str.split().str.len()
    df['sentence_count'] = df['text'].str.count(r'[.!?]+').replace(0, 1)
    df['avg_sentence_length'] = (
        (df['word_count']/df['sentence_count']).clip(upper=100))
    df['punct_count'] = df['text'].str.count(r'[^\w\s]')
    df['punct_ratio'] = (
        (df['punct_count']/df['text_length']).clip(0, 0.3))
    df['ttr'] = df['text'].apply(_ttr)
    df['digit_ratio'] = df['text'].str.count(r'\d') / (
        df['text_length'].replace(0, 1))
    df['upper_ratio'] = df['text'].str.count(r'[A-Z]') / (
        df['text_length'].replace(0, 1))
    df['bangs'] = df['text'].str.count(r'!')
    df['questions'] = df['text'].str.count(r'\?')
    return df


## Binary features and 2^3 sweep

In [6]:
def ends_with_letter(text: str) -> int:
    s = text.rstrip();
    return int(len(s) > 0 and s[-1] in string.ascii_letters)
def has_5gram_repetition(text: str) -> int:
    toks = re.findall(r"\S+", text)
    if len(toks) < 10: return 0
    seen = {}; w = 5
    for i in range(len(toks) - w + 1):
        key = tuple(toks[i:i+w])
        if key in seen: return 1
        seen[key] = 1
    return 0
COMMON_SMALL = set(['the','be','to','of','and','a','in','that','have',
    'i','it','for','not','on','with','he','as','you','do','at','this',
    'but','his'])
def max_uncommon_binary(text: str, thr_rep: int = 3,
                        thr_count: int = 5) -> int:
    toks = [t.lower() for t in re.findall(r"\w+", text)]
    if not toks: return 0
    freqs = {}; uncommon = 0
    for t in toks:
        if t not in COMMON_SMALL:
            uncommon += 1
            freqs[t] = freqs.get(t, 0) + 1
    if uncommon < thr_count: return 0
    return int(any(v >= thr_rep for v in freqs.values()))
def add_binary_feats(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy(); tqdm.pandas()
    out['ends_with_letter'] = out['text'].progress_apply(ends_with_letter)
    out['has_5gram_repetition'] = out['text'].progress_apply(
        has_5gram_repetition)
    out['max_uncommon_binary'] = out['text'].progress_apply(
        max_uncommon_binary)
    return out
def sweep_binary_subsets(y_true: np.ndarray, fe_df: pd.DataFrame):
    cols = ['ends_with_letter','has_5gram_repetition','max_uncommon_binary']
    best_f1, best_key = -1.0, 'none'
    for mask in range(1, 1 << len(cols)):
        sel = [cols[i] for i in range(len(cols)) if (mask >> i) & 1]
        rule = fe_df[sel].any(axis=1).astype(int).values
        f1 = f1_score(y_true, rule)
        key = '|'.join(sel)
        if f1 > best_f1: best_f1, best_key = f1, key
    return best_key, float(best_f1)


## Load data

In [7]:
train = process_text_file('train.csv')
val = process_text_file('val.csv')
test = process_text_file('test.csv')
assert 'label' in train.columns and 'label' in val.columns
assert 'label' not in test.columns and 'id' in test.columns
print('Rows:', len(train), len(val), len(test))


Rows: 319071 56792 60743


## Sentiment + binaries

In [8]:
train = _sentiment(train); val = _sentiment(val); test = _sentiment(test)
train = add_binary_feats(train); val = add_binary_feats(val);
test = add_binary_feats(test)
rk, rf1 = sweep_binary_subsets(val['label'].astype(int).values, val)
print(f'Best binary subset (val): {rk} | F1={rf1:.4f}')


  0%|          | 0/319071 [00:00<?, ?it/s]

 37%|███▋      | 117793/319071 [00:00<00:00, 1177928.31it/s]

 79%|███████▊  | 251186/319071 [00:00<00:00, 1269687.55it/s]

100%|██████████| 319071/319071 [00:00<00:00, 1057220.75it/s]

  0%|          | 0/319071 [00:00<?, ?it/s]

...

 95%|█████████▌| 57823/60743 [00:05<00:00, 13700.23it/s]

 98%|█████████▊| 59328/60743 [00:05<00:00, 12035.76it/s]

100%|█████████▉| 60645/60743 [00:06<00:00, 11267.92it/s]

100%|██████████| 60743/60743 [00:06<00:00, 10019.64it/s]

Best binary subset (val): max_uncommon_binary | F1=0.6240


## Numeric + TF–IDF design

In [9]:
num_cols = ['text_length','word_count','ttr','sentence_count',
            'avg_sentence_length','punct_ratio','sentiment_polarity',
            'sentiment_subjectivity','digit_ratio','upper_ratio','bangs',
            'questions','ends_with_letter','has_5gram_repetition',
            'max_uncommon_binary']
Xtr_num = csr_matrix(train[num_cols].values)
Xva_num = csr_matrix(val[num_cols].values)
Xte_num = csr_matrix(test[num_cols].values)
vec_word = TfidfVectorizer(ngram_range=(1, 3), max_features=50000,
                           min_df=2, stop_words='english')
Xtr_w = vec_word.fit_transform(train['text'])
Xva_w = vec_word.transform(val['text'])
Xte_w = vec_word.transform(test['text'])
X_train = hstack([Xtr_num, Xtr_w])
X_val = hstack([Xva_num, Xva_w])
X_test = hstack([Xte_num, Xte_w])
y_train = train['label'].astype(int).values
y_val = val['label'].astype(int).values
print('Shapes:', X_train.shape, X_val.shape, X_test.shape)


Shapes: (319071, 50015) (56792, 50015) (60743, 50015)


## Tuning and calibration

In [ ]:
def _xgb_space():
    return {'n_estimators':[300,500,700],'max_depth':[4,6,8],
            'learning_rate':[0.05,0.1],'min_child_weight':[1,3],
            'subsample':[0.7,1.0],'colsample_bytree':[0.7,1.0],
            'reg_alpha':[0.0,0.1,0.5],'reg_lambda':[0.5,1.0,1.5]}
def _lr_space():
    return {'C':[0.5,1.0,2.0,4.0],'penalty':['l2'],
            'solver':['liblinear','lbfgs'],'class_weight':[None,'balanced']}
def tune_xgb(X, y):
    base = XGBClassifier(random_state=42, eval_metric='logloss',
                         tree_method='hist', n_jobs=1)
    rs = RandomizedSearchCV(base, _xgb_space(), n_iter=25, scoring='f1',
                            n_jobs=1, cv=5, verbose=1, random_state=42,
                            refit=True)
    rs.fit(X, y); print('Best XGB:', rs.best_params_); return rs.best_estimator_
def tune_lr(X, y):
    base = LogisticRegression(max_iter=2000, random_state=42)
    rs = RandomizedSearchCV(base, _lr_space(), n_iter=25, scoring='f1',
                            n_jobs=1, cv=5, verbose=1, random_state=42,
                            refit=True)
    rs.fit(X, y); print('Best LR:', rs.best_params_); return rs.best_estimator_
xgb_tuned = tune_xgb(X_train, y_train)
lr_tuned = tune_lr(X_train, y_train)
X_trval = vstack([X_train, X_val]); y_trval = np.concatenate([y_train, y_val])
xgb_tuned.fit(X_trval, y_trval); lr_tuned.fit(X_trval, y_trval)
cal_xgb = CalibratedClassifierCV(xgb_tuned, method='sigmoid', cv='prefit')
cal_xgb.fit(X_val, y_val)
cal_lr = CalibratedClassifierCV(lr_tuned, method='sigmoid', cv='prefit')
cal_lr.fit(X_val, y_val)


## Ensembling, thresholding, prevalence match

In [ ]:
def decode_prevalence(y_prob: np.ndarray, pos_rate: float) -> np.ndarray:
    n = len(y_prob); k = int(round(pos_rate * n))
    idx = np.argsort(-y_prob); out = np.zeros(n, dtype=int); out[idx[:k]] = 1
    return out
p_xgb = cal_xgb.predict_proba(X_val)[:, 1]
p_lr = cal_lr.predict_proba(X_val)[:, 1]
best_w, best_f1, best_thr = 0.5, -1.0, 0.5
for w in np.linspace(0.0, 1.0, 21):
    p = w * p_xgb + (1.0 - w) * p_lr
    for thr in np.arange(0.1, 0.91, 0.01):
        f1 = f1_score(y_val, (p >= thr).astype(int))
        if f1 > best_f1: best_w, best_f1, best_thr = float(w), float(f1), float(thr)
print(f'Threshold head: w={best_w:.2f} thr={best_thr:.2f} F1={best_f1:.4f}')
p_ens = best_w * p_xgb + (1.0 - best_w) * p_lr
val_pos_rate = float(np.mean(y_val))
yhat_topk = decode_prevalence(p_ens, val_pos_rate)
f1_topk = f1_score(y_val, yhat_topk)
print(f'Prevalence head: rate={val_pos_rate:.3f} F1={f1_topk:.4f}')
rk, rf1 = sweep_binary_subsets(y_val, val)
print(f'Rule head (best subset {rk}) F1={rf1:.4f}')
heads = [('threshold', best_f1), ('prevalence', f1_topk), ('rule', rf1)]
heads.sort(key=lambda x: x[1], reverse=True)
print('Head ranking:', heads)


## Validation diagnostics

In [ ]:
winner = heads[0][0]
if winner == 'threshold': yhat_val = (p_ens >= best_thr).astype(int)
elif winner == 'prevalence': yhat_val = yhat_topk
else:
    yhat_val = val[['ends_with_letter','has_5gram_repetition',
                    'max_uncommon_binary']].any(axis=1).astype(int).values
print(classification_report(y_val, yhat_val))
residual_plot(y_val, p_ens, 'Residuals: ensemble on val')
qq_plot(y_val - p_ens, 'QQ: residuals (val)')
plot_roc_pr(y_val, p_ens, 'Validation ROC/PR (ensemble)')
plot_confusion(y_val, yhat_val, 'Confusion (val, winner head)')


## Predict test and save submission

In [ ]:
p_xgb_te = cal_xgb.predict_proba(X_test)[:, 1]
p_lr_te = cal_lr.predict_proba(X_test)[:, 1]
p_ens_te = best_w * p_xgb_te + (1.0 - best_w) * p_lr_te
if winner == 'threshold': yhat_te = (p_ens_te >= best_thr).astype(int)
elif winner == 'prevalence': yhat_te = decode_prevalence(p_ens_te, val_pos_rate)
else:
    yhat_te = test[['ends_with_letter','has_5gram_repetition',
                    'max_uncommon_binary']].any(axis=1).astype(int).values
submission = pd.DataFrame({'id': test['id'], 'label': yhat_te})
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv with', len(submission), 'rows')
